# Kafka Connectivity Test

Verifies that Databricks can reach Confluent Kafka using credentials read from the Azure Key Vault-backed secret scope `kv-fraud-detection`. Runs in three levels: network, authentication and topic access, and data. No secret values are printed. Clear outputs before committing.

In [0]:
# Non-secret configuration
SECRET_SCOPE = "kv-fraud-detection"
KAFKA_TOPIC = "credit_card_transactions"

# Secrets are read from the Azure Key Vault-backed scope; never print these values
bootstrap_servers = dbutils.secrets.get(SECRET_SCOPE, "kafka-bootstrap-servers")

In [0]:
import socket

# Level 1: can this compute open a TCP connection to the Confluent bootstrap server?
# Only the result is printed, not the host name.
results = []
for endpoint in bootstrap_servers.split(","):
    host, port = endpoint.strip().rsplit(":", 1)
    try:
        with socket.create_connection((host, int(port)), timeout=10):
            results.append(("OK", int(port), None))
    except Exception as e:
        results.append(("FAILED", int(port), type(e).__name__))

for i, (status, port, err) in enumerate(results, start=1):
    print(f"Bootstrap endpoint {i} on port {port}: {status}" + (f" ({err})" if err else ""))

print("Level 1 PASSED" if all(r[0] == "OK" for r in results) else "Level 1 FAILED")

In [0]:
from pyspark.sql import functions as F

# Level 2: log in to Confluent with the Key Vault credentials and read the topic once (batch, no stream, no checkpoint).
# The login string is built inline in the options and never stored or printed.
kafka_df = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", KAFKA_TOPIC)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        f'username="{dbutils.secrets.get(SECRET_SCOPE, "kafka-api-key")}" '
        f'password="{dbutils.secrets.get(SECRET_SCOPE, "kafka-api-secret")}";',
    )
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

# Only metadata is shown: messages per partition and offset range
partition_summary = (
    kafka_df.groupBy("partition")
    .agg(
        F.count("*").alias("message_count"),
        F.min("offset").alias("min_offset"),
        F.max("offset").alias("max_offset"),
    )
    .orderBy("partition")
)
display(partition_summary)

total_messages = kafka_df.count()
print(f"Level 2 PASSED: authenticated and read topic '{KAFKA_TOPIC}' ({total_messages} messages available)")

In [0]:
# Level 3: does the data look right?
# Kafka stores key and value as raw bytes; cast them to text to inspect a small sample of the latest messages.
sample_df = (
    kafka_df.select(
        "partition",
        "offset",
        "timestamp",
        F.col("key").cast("string").alias("key"),
        F.col("value").cast("string").alias("value"),
    )
    .orderBy(F.col("timestamp").desc())
    .limit(10)
)
display(sample_df)

# Basic quality signals across all messages read in Level 2
quality = kafka_df.agg(
    F.count("*").alias("total_messages"),
    F.sum(F.when(F.col("value").isNull(), 1).otherwise(0)).alias("null_values"),
    F.min("timestamp").alias("oldest_message"),
    F.max("timestamp").alias("newest_message"),
)
display(quality)